# BERT 전이 학습

In [15]:
from tensorflow.keras.utils import get_file

# txt 파일을 다운로드하고 로컬 저장 경로 반환
ratings_train_path = get_file('ratings_train.txt','https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt')
ratings_test_path = get_file('ratings_test.txt','https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt')


In [16]:
import pandas as pd
# 데이터(csv) 로드 -> DataFrame 변환
ratings_train_df = pd.read_csv(ratings_train_path, sep='\t')
ratings_test_df = pd.read_csv(ratings_test_path, sep='\t')

display(ratings_train_df.head())
display(ratings_test_df.head())

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [17]:
print(ratings_train_df.isnull().sum())
print(ratings_test_df.isnull().sum())

id          0
document    5
label       0
dtype: int64
id          0
document    3
label       0
dtype: int64


In [18]:
ratings_train_df = ratings_train_df.dropna(how='any')
ratings_test_df = ratings_test_df.dropna(how='any')

In [19]:
print(ratings_train_df.info())
print(ratings_test_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        149995 non-null  int64 
 1   document  149995 non-null  object
 2   label     149995 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.6+ MB
None
<class 'pandas.core.frame.DataFrame'>
Index: 49997 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        49997 non-null  int64 
 1   document  49997 non-null  object
 2   label     49997 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.5+ MB
None


In [20]:
ratings_train_df = ratings_train_df.sample(n= 15000, random_state= 0)
ratings_test_df = ratings_test_df.sample(n= 5000, random_state= 0)

ratings_train_df['label'].value_counts(), ratings_test_df['label'].value_counts()

(label
 0    7512
 1    7488
 Name: count, dtype: int64,
 label
 0    2532
 1    2468
 Name: count, dtype: int64)

In [21]:
# 텍스트 / 라벨을 리스트로 변환
X_train = ratings_train_df['document'].values.tolist()
y_train = ratings_train_df['label'].values.tolist()


X_test = ratings_test_df['document'].values.tolist()
y_test = ratings_test_df['label'].values.tolist()



# 토크나이저 / 모델 준비
- bert 한국어 버전 사전학습 모델 klue/ bert-base

In [22]:
from transformers import AutoModel, AutoTokenizer
from transformers import BertForSequenceClassification

model_name = 'klue/bert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)   # 토크나이저 (토큰화 규칙/ 어휘사전)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels= 2)         # 모델 로드 (가중치 다운로드 캐시에 저장)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
# 데이터 문장들 토큰화 + 패딩/ 자르기 처리
X_train = tokenizer(X_train, padding= True, truncation = True, return_tensors= 'pt')
X_test = tokenizer(X_test, padding= True, truncation = True, return_tensors= 'pt')

X_train[:3], X_test[:3]

([Encoding(num_tokens=142, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=142, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=142, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])],
 [Encoding(num_tokens=118, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=118, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=118, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])])

In [ ]:
print(X_train['input_ids'][0])      # 토큰이 정수 ID로 변환된 시퀀스
print(X_train['attention_mask'][0]) # 실제 토큰 = 1, 패딩 = 0 으로 구분한 마스크 
print(X_train['token_type_ids'][0]) # 문장 구분 ID()

tensor([    2,  1800,  2178,   860,  3629, 16516,  2031,    18,    18,    18,
        14242,  2205,  2062,     3,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

### 데이터 파이프라인 생성

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Bert 입력 데이터와 라벨을 함께 관리하는 Dataset 클래스
class NSMCDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings      # 토큰화 결과 (input_ids, attention_mask 등 ) 저장
        self.labels = labels            # 정답 라벨
        
    def __len__(self):
        return len(self.labels)
    
    # idx 번째 샘플의 토큰화 결과 (input_ids, attention_mask 등)을 딕셔너리 형태로 반환
    def __getitem__(self, idx):
        item = {key: value[idx] for key, value in self.encodings.items()}
        
        # idx번째 정답 라벨을 Pytorch 정수형 LongTensor형태로 변환해서 추가
        item['labels'] = torch.tensor(self.labels[idx], dtype= torch.long)
        
        return item

In [ ]:
train_dataset = NSMCDataset(X_train, y_train)
test_dataset = NSMCDataset(X_test, y_test)

# 학습 데이터 : 셔플 후 64개씩 배치/ 테스트 데이터 : 셔플없이 64 개씩 배치
train_dataloader = DataLoader(train_dataset, batch_size= 64, shuffle= True)
test_dataloader = DataLoader(test_dataset, batch_size= 64, shuffle= False)

In [ ]:
# 파인튜닝 설정
import torch
from transformers import get_scheduler

epochs = 5

# AdamW 최적화 함수 : 가중치 감쇠 적용(과적합 완화)
optimizer = torch.optim.AdamW(model.parameters(), lr= 5e-5, weight_decay = 0.1)

num_train_steps = len(train_dataloader) *epochs    # 전체 학습 step 수

# warmup step수 : 전체 학습 step에서 10% 는 학습률을 점진적으로 증가
# - 사전학습 fine-tuning 시에는 초반에 LR가 크면 학습이 불안정해지는 경우가 많아서 0에서부터 lr까지 점차적으로 학습률을 늘려준다.
num_warmup_steps = int(num_train_steps * 0.1)

lr_scheduler = get_scheduler(
    name = 'linear',            # Warmup 이후 학습률 선형적으로 감소
    optimizer= optimizer,
    num_warmup_steps= num_warmup_steps,
    num_training_steps= num_train_steps
)

In [ ]:
# 학습
import torch.nn as nn
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

for epoch in tqdm(range(epochs)):
    model.train()
    
    total_loss = 0
    
    for batch in tqdm(train_dataloader, desc= f'{epoch+1}/{epochs}', leave= False):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        
        lr_scheduler.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss/ len(train_dataloader)
    print(f'Epoch : {epoch+1 } / {epochs} : Loss {avg_loss:.4f}')

  0%|          | 0/5 [00:00<?, ?it/s]

1/5:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch : 1 / 5 : Loss 0.0000


2/5:   0%|          | 0/235 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
model.save_pretrained('nsmc_model/bert-base')       # 학습된 모델 저장
tokenizer.save_pretrained('nsmc_model/bert-base')   # 모델에서 사용한 토크나이저 저장

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('nsmc_model/bert-base/tokenizer_config.json',
 'nsmc_model/bert-base/tokenizer.json')

In [24]:
# 감성분석 파이프라인 생성
from transformers import TextClassificationPipeline

# 입력 텍스트 -> 토큰화 -> 모델추론 -> 라벨/점수 반환 파이프라인
sentiment_classifier = TextClassificationPipeline(
    tokenizer= tokenizer,
    model= model, 
    framework= 'pt',    # Pytorch 기반 모델
    top_k = None        # 모든 클래스 라벨과 확률 반환
)


In [25]:
model.config.id2label = {
    0 : '부정',
    1 : '긍정'
}

In [26]:
sentiment_classifier('이것은 제 생에 가장 훌륭한 영화입니다!! 대단합니다!')

[[{'label': '긍정', 'score': 0.5022179484367371},
  {'label': '부정', 'score': 0.49778199195861816}]]

# HuggingFace 업로드

In [28]:
from getpass import getpass         # 입력값을 화면에 표시하지 않고 바로받는 함수
from huggingface_hub import login   # HuggingFace 로그인

HF_TOKEN = getpass('HF_TOKEN :')    # AccessToken 입력
login(token= HF_TOKEN)              # 입력한 토큰으로 로그인

In [30]:
REPO_NAME = 'bert-base-nsmc'

# 학습된 모델과 토크나이저를 Hub Repo 업로드 (임시폴더 사용, 토큰으로 인증)
model.push_to_hub(REPO_NAME,token= HF_TOKEN)
tokenizer.push_to_hub(REPO_NAME, token= HF_TOKEN)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._ayv0ja/model.safetensors:   4%|3         | 16.1MB /  442MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/anstjdghdyd/bert-base-nsmc/commit/7f6443d709da7e246ba3a4572db5ecf1a0c35ae4', commit_message='Upload tokenizer', commit_description='', oid='7f6443d709da7e246ba3a4572db5ecf1a0c35ae4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/anstjdghdyd/bert-base-nsmc', endpoint='https://huggingface.co', repo_type='model', repo_id='anstjdghdyd/bert-base-nsmc'), pr_revision=None, pr_num=None)

In [32]:
# hugging face 에 업로드한 내 토크나이저/ 모델을 가져옴
from transformers import AutoTokenizer, AutoModelForSequenceClassification

HUB_NAME = 'anstjdghdyd/bert-base-nsmc'

tokenizer = AutoTokenizer.from_pretrained(HUB_NAME)
model = AutoModelForSequenceClassification.from_pretrained(HUB_NAME)

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/404 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [33]:
from transformers import pipeline

sentiment_classifier = pipeline('text-classification', model = HUB_NAME)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [42]:
sentiment_classifier([
    '한국 영화는 이래서 안돼~',
    '역시 봉감독이 최고야!',
    '오디세이급이야',
    '드럽게 재미없네'])

[{'label': '부정', 'score': 0.5422670245170593},
 {'label': '긍정', 'score': 0.5851829648017883},
 {'label': '긍정', 'score': 0.5547667741775513},
 {'label': '긍정', 'score': 0.6144134998321533}]